# LeetCode 2245 — Maximum Trailing Zeros in a Cornered Path bằng Python

Link đề: https://leetcode.com/problems/maximum-trailing-zeros-in-a-cornered-path/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxTrailingZeros(self, grid: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu **số chữ số 0 tận cùng** của một tích bằng `min(số thừa số 2, số thừa số 5)` — và vì sao không được nhân trực tiếp rồi đếm.
2. Nhận ra tính **đơn điệu**: mọi ô đều dương nên kéo dài đường đi chỉ làm số thừa số 2 và 5 **không giảm** ⇒ đường tối ưu luôn **chạm biên** ở cả hai đầu.
3. Rút gọn mọi "đường có tối đa một khúc quẹo" thành **4 dạng chữ L** quanh một ô góc: (trái hoặc phải) × (lên hoặc xuống).
4. Dùng **tổng tiền tố theo hàng và theo cột** của số thừa số 2 và 5 để tính mỗi chữ L trong O(1) ⇒ tổng O(m·n).
5. Viết bộ vét cạn **liệt kê thật mọi đường có góc** để kiểm chứng trên lưới nhỏ.

## 2. Đề bài & ví dụ

Cho lưới `grid` `m × n` gồm các số nguyên dương. Một **đường có góc** (cornered path) là dãy ô kề nhau với **tối đa một lần quẹo**: đi thẳng theo một phương (ngang **hoặc** dọc) tới chỗ quẹo (nếu có), rồi đi thẳng theo phương còn lại; không đi lại ô đã qua.

**Tích** của đường = tích mọi giá trị trên đường. Trả về **số chữ số 0 tận cùng lớn nhất** của tích của một đường có góc.

**Ví dụ 1:** `grid = [[23,17,15,3,20],[8,1,20,27,11],[9,4,6,2,21],[40,9,1,10,6],[22,7,4,5,3]]` → `3` (một đường: `15·20·6·1·10 = 18000`).

**Ví dụ 2:** `grid = [[4,3,2],[7,6,1],[8,8,8]]` → `0` (không có thừa số 5 nào).

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`, `1 ≤ m·n ≤ 10⁵`, `1 ≤ grid[i][j] ≤ 1000`.

Lưu ý: "ngang" gồm cả trái lẫn phải, "dọc" gồm cả lên lẫn xuống; một ô đơn lẻ cũng là đường hợp lệ.

In [1]:
from typing import List
import random, time

examples = [
    ([[23, 17, 15, 3, 20], [8, 1, 20, 27, 11], [9, 4, 6, 2, 21], [40, 9, 1, 10, 6], [22, 7, 4, 5, 3]], 3),
    ([[4, 3, 2], [7, 6, 1], [8, 8, 8]], 0),
]
def trailing_zeros(x):
    c = 0
    while x % 10 == 0:
        x //= 10; c += 1
    return c
print(15 * 20 * 6 * 1 * 10, trailing_zeros(15 * 20 * 6 * 1 * 10))
for g, exp in examples:
    print("kỳ vọng", exp)
    for row in g: print("   ", row)

18000 3
kỳ vọng 3
    [23, 17, 15, 3, 20]
    [8, 1, 20, 27, 11]
    [9, 4, 6, 2, 21]
    [40, 9, 1, 10, 6]
    [22, 7, 4, 5, 3]
kỳ vọng 0
    [4, 3, 2]
    [7, 6, 1]
    [8, 8, 8]


## 3. Kiến thức nền

### 3.1 Số 0 tận cùng = min(số thừa số 2, số thừa số 5)

`10 = 2 · 5`. Một tích `P = 2^a · 5^b · k` (với `k` không chia hết cho 2 và 5) có đúng `min(a, b)` chữ số 0 tận cùng. Vì phân tích thừa số nguyên tố là cộng tính theo phép nhân, số thừa số 2 (hay 5) của một tích bằng **tổng** số thừa số 2 (hay 5) của các thừa số.

In [2]:
def factor_count(x, p):
    c = 0
    while x % p == 0:
        x //= p; c += 1
    return c

vals = [15, 20, 6, 1, 10]
a = sum(factor_count(v, 2) for v in vals); b = sum(factor_count(v, 5) for v in vals)
print("số thừa số 2 =", a, "| số thừa số 5 =", b, "| min =", min(a, b))

số thừa số 2 = 4 | số thừa số 5 = 3 | min = 3


### 3.2 Đừng tính tích thật

Tích của đường dài `m + n − 1` ô với giá trị tới 1000 có tới ~`3(m + n)` chữ số — Python làm được (số nguyên lớn) nhưng **rất chậm** khi phải làm cho hàng triệu đường; ngôn ngữ khác thì tràn số ngay. Chỉ cần **hai bộ đếm** (2 và 5).

In [3]:
prod = 1
for _ in range(2000): prod *= 997
print("tích 2000 số 997 có khoảng", int(prod.bit_length() * 0.30103) + 1, "chữ số (ước lượng qua số bit;",
      "Python còn chặn str() với số > 4300 chữ số)")

tích 2000 số 997 có khoảng 5998 chữ số (ước lượng qua số bit; Python còn chặn str() với số > 4300 chữ số)


### 3.3 Tổng tiền tố theo hàng và theo cột

`R[i][j]` = tổng của `a[i][0..j−1]` (tiền tố hàng), `Cc[i][j]` = tổng của `a[0..i−1][j]` (tiền tố cột). Tổng một đoạn ngang `a[i][l..r]` = `R[i][r+1] − R[i][l]`; đoạn dọc `a[t..b][j]` = `Cc[b+1][j] − Cc[t][j]`.

In [4]:
a = [[1, 2, 3], [4, 5, 6]]
m, n = 2, 3
R = [[0] * (n + 1) for _ in range(m)]
Cc = [[0] * n for _ in range(m + 1)]
for i in range(m):
    for j in range(n):
        R[i][j + 1] = R[i][j] + a[i][j]
        Cc[i + 1][j] = Cc[i][j] + a[i][j]
print("R =", R); print("Cc =", Cc)
print("tổng a[1][0..2] =", R[1][3] - R[1][0], "| tổng a[0..1][2] =", Cc[2][2] - Cc[0][2])

R = [[0, 1, 3, 6], [0, 4, 9, 15]]
Cc = [[0, 0, 0], [1, 2, 3], [5, 7, 9]]
tổng a[1][0..2] = 15 | tổng a[0..1][2] = 9


### 3.4 Đường có góc = "chữ L" quanh ô góc

Mọi đường có góc gồm một **ô góc** `(i, j)`, một **cánh ngang** (một phía: trái hoặc phải, có thể rỗng) và một **cánh dọc** (một phía: lên hoặc xuống, có thể rỗng). Đường thẳng là trường hợp một cánh rỗng (ô góc là đầu mút).

## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — đơn điệu.** Mọi giá trị ≥ 1 ⇒ số thừa số 2 và 5 của mỗi ô ≥ 0. Kéo dài một cánh thêm một ô **không bao giờ giảm** `a` (tổng số 2) hay `b` (tổng số 5) ⇒ không giảm `min(a, b)`. Vậy với ô góc và hướng của hai cánh đã chọn, **kéo mỗi cánh tới tận biên** luôn tốt nhất.

**Quan sát 2 — chỉ còn 4 ứng viên mỗi ô.** Với ô góc `(i, j)`: cánh ngang là "toàn bộ phía trái" hoặc "toàn bộ phía phải" của hàng `i`; cánh dọc là "toàn bộ phía trên" hoặc "toàn bộ phía dưới" của cột `j`. ⇒ 4 chữ L: trái+trên, trái+dưới, phải+trên, phải+dưới. Tổng cộng `4·m·n` ứng viên.

**Quan sát 3 — đường thẳng đã được bao hàm.** Đường thẳng ngang từ `(i, l)` tới `(i, r)` ≤ (theo quan sát 1) đường thẳng tới biên, tức chữ L có ô góc ở một đầu và cánh dọc rỗng — mà kéo dài cánh dọc ra biên chỉ tốt hơn. Vậy 4 chữ L "đầy đủ" ở mọi ô đã đủ.

**Tính tay** một phần ví dụ 1 tại ô góc `(3, 3)` (giá trị 10, đánh số từ 0): hàng 3 là `[40, 9, 1, 10, 6]`, cột 3 là `[3, 27, 2, 10, 5]`.
- Cánh trái `40·9·1` = `2³·5 · 9` ⇒ (2: 3, 5: 1); ô góc 10 ⇒ (1, 1); cánh trên `3·27·2` ⇒ (1, 0).
- Trái+trên: 2 → 3 + 1 + 1 = 5; 5 → 1 + 1 + 0 = 2 ⇒ `min = 2`.
- Phải+dưới: cánh phải `6` (1, 0), cánh dưới `5` (0, 1) ⇒ 2: 1 + 1 + 0 = 2; 5: 1 + 0 + 1 = 2 ⇒ `min = 2`.

In [5]:
g = examples[0][0]
def counts(vals):
    return sum(factor_count(v, 2) for v in vals), sum(factor_count(v, 5) for v in vals)
i, j = 3, 3
row, col = g[i], [g[r][j] for r in range(len(g))]
arms = {"trái": row[:j], "phải": row[j + 1:], "trên": col[:i], "dưới": col[i + 1:]}
corner = counts([g[i][j]])
for h in ["trái", "phải"]:
    for v in ["trên", "dưới"]:
        a2 = counts(arms[h])[0] + corner[0] + counts(arms[v])[0]
        a5 = counts(arms[h])[1] + corner[1] + counts(arms[v])[1]
        print(f"{h}+{v}: cánh ngang {arms[h]}, cánh dọc {arms[v]} -> (2:{a2}, 5:{a5}) -> {min(a2, a5)} số 0")

trái+trên: cánh ngang [40, 9, 1], cánh dọc [3, 27, 2] -> (2:5, 5:2) -> 2 số 0
trái+dưới: cánh ngang [40, 9, 1], cánh dọc [5] -> (2:4, 5:3) -> 3 số 0
phải+trên: cánh ngang [6], cánh dọc [3, 27, 2] -> (2:3, 5:1) -> 1 số 0
phải+dưới: cánh ngang [6], cánh dọc [5] -> (2:2, 5:2) -> 2 số 0


## 5. Cách 1 — Brute Force / baseline: liệt kê thật mọi đường có góc

**Ý tưởng:** với mỗi ô xuất phát, mỗi hướng ban đầu `d1` (4 hướng), mỗi độ dài `L1 ≥ 0`, rồi quẹo sang một trong 2 hướng vuông góc `d2` với mỗi độ dài `L2 ≥ 0`: cộng dồn số thừa số 2, 5. Đây là định nghĩa của đề, không dùng quan sát nào.

In [6]:
DIRS = [(0, 1), (0, -1), (1, 0), (-1, 0)]
def brute(grid):
    m, n = len(grid), len(grid[0])
    c2 = [[factor_count(x, 2) for x in row] for row in grid]
    c5 = [[factor_count(x, 5) for x in row] for row in grid]
    best = 0
    for si in range(m):
        for sj in range(n):
            for d1 in DIRS:
                # đi L1 bước theo d1 (gồm cả ô xuất phát)
                i, j = si, sj; a, b = c2[i][j], c5[i][j]
                while True:
                    best = max(best, min(a, b))
                    for d2 in DIRS:
                        if d2[0] * d1[0] + d2[1] * d1[1] != 0: continue      # chỉ hướng vuông góc
                        x, y = i + d2[0], j + d2[1]; aa, bb = a, b
                        while 0 <= x < m and 0 <= y < n:
                            aa += c2[x][y]; bb += c5[x][y]
                            best = max(best, min(aa, bb))
                            x, y = x + d2[0], y + d2[1]
                    i, j = i + d1[0], j + d1[1]
                    if not (0 <= i < m and 0 <= j < n): break
                    a += c2[i][j]; b += c5[i][j]
    return best

for g, exp in examples:
    got = brute(g)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp:** `m·n` ô xuất phát × 4 hướng × O(m + n) độ dài × O(m + n) cánh thứ hai ⇒ **O(m·n·(m+n)²)** — với `m = n ≈ 316` là ~10¹⁰ ⇒ không khả thi.

**Nút thắt:** (1) thử mọi độ dài cánh, trong khi theo quan sát 1 chỉ cần cánh dài nhất; (2) cộng dồn lại từng ô cho mỗi cánh, trong khi tổng tiền tố cho O(1).

### 5.1 Baseline thứ hai: 4 chữ L đầy đủ nhưng cộng dồn trực tiếp — O(m·n·(m+n))

In [7]:
def four_L_direct(grid):
    m, n = len(grid), len(grid[0])
    c2 = [[factor_count(x, 2) for x in row] for row in grid]
    c5 = [[factor_count(x, 5) for x in row] for row in grid]
    best = 0
    for i in range(m):
        for j in range(n):
            L = (sum(c2[i][:j]), sum(c5[i][:j]));  Rr = (sum(c2[i][j + 1:]), sum(c5[i][j + 1:]))
            U = (sum(c2[r][j] for r in range(i)), sum(c5[r][j] for r in range(i)))
            D = (sum(c2[r][j] for r in range(i + 1, m)), sum(c5[r][j] for r in range(i + 1, m)))
            for h in (L, Rr):
                for v in (U, D):
                    best = max(best, min(h[0] + v[0] + c2[i][j], h[1] + v[1] + c5[i][j]))
    return best

for g, exp in examples:
    print(four_L_direct(g), "OK" if four_L_direct(g) == exp else "SAI")

3 OK
0 OK


Đã bỏ được việc thử mọi độ dài, nhưng mỗi cánh vẫn tốn O(m + n) ⇒ O(m·n·(m+n)), worst ~6·10⁷ với lưới vuông — chậm trong Python.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** mỗi cánh là một **đoạn liên tiếp** của một hàng hoặc một cột ⇒ tổng số thừa số 2/5 trên cánh tính bằng **hiệu tiền tố** O(1).

Ký hiệu (chỉ số từ 0): với ma trận đếm `t ∈ {c2, c5}`:
- `left_t(i, j)` = Σ `t[i][0..j−1]` = `R_t[i][j]`.
- `right_t(i, j)` = Σ `t[i][j+1..n−1]` = `R_t[i][n] − R_t[i][j+1]`.
- `up_t(i, j)` = Σ `t[0..i−1][j]` = `C_t[i][j]`.
- `down_t(i, j)` = Σ `t[i+1..m−1][j]` = `C_t[m][j] − C_t[i+1][j]`.

Chữ L `(h, v)` tại `(i, j)`: `min(h_2 + v_2 + c2[i][j], h_5 + v_5 + c5[i][j])` — ô góc tính **đúng một lần**.

**Quy tắc quan trọng:** lấy `min` **sau khi cộng** tổng số 2 và tổng số 5 của cả đường — không được lấy `min` cho từng cánh rồi cộng (Lỗi 2).

**Bất biến:** sau khi dựng tiền tố, mọi đại lượng `left/right/up/down` đúng bằng tổng trên cánh tương ứng (không gồm ô góc).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố hàng/cột riêng cho 2 và 5 (cách chuẩn)

In [8]:
def prefix_solution(grid):
    m, n = len(grid), len(grid[0])
    c2 = [[factor_count(x, 2) for x in row] for row in grid]
    c5 = [[factor_count(x, 5) for x in row] for row in grid]
    R2 = [[0] * (n + 1) for _ in range(m)]; R5 = [[0] * (n + 1) for _ in range(m)]
    C2 = [[0] * n for _ in range(m + 1)]; C5 = [[0] * n for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            R2[i][j + 1] = R2[i][j] + c2[i][j]; R5[i][j + 1] = R5[i][j] + c5[i][j]
            C2[i + 1][j] = C2[i][j] + c2[i][j]; C5[i + 1][j] = C5[i][j] + c5[i][j]
    best = 0
    for i in range(m):
        for j in range(n):
            l2, l5 = R2[i][j], R5[i][j]
            r2, r5 = R2[i][n] - R2[i][j + 1], R5[i][n] - R5[i][j + 1]
            u2, u5 = C2[i][j], C5[i][j]
            d2, d5 = C2[m][j] - C2[i + 1][j], C5[m][j] - C5[i + 1][j]
            x2, x5 = c2[i][j], c5[i][j]
            for h2, h5 in ((l2, l5), (r2, r5)):
                for v2, v5 in ((u2, u5), (d2, d5)):
                    z = min(h2 + v2 + x2, h5 + v5 + x5)
                    if z > best: best = z
    return best

for g, exp in examples:
    got = prefix_solution(g)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp cách 2:** O(m·n·log V) để đếm thừa số (mỗi số ≤ 1000 có ≤ 9 thừa số 2, ≤ 4 thừa số 5) + O(m·n) dựng tiền tố + O(m·n) duyệt 4 chữ L ⇒ **O(m·n)**; bộ nhớ O(m·n).

### 7.2 Cách 3 — Bốn mảng "cánh tích lũy" theo hướng (không dùng hiệu)

Thay vì tiền tố + hiệu, tính trực tiếp `L[i][j]` = tổng từ biên trái tới `(i, j)` **bao gồm** ô `(i, j)`, tương tự `Rg, U, D`. Khi đó chữ L trái+trên tại `(i, j)` = `L[i][j] + U[i][j] − ô(i, j)` (trừ ô góc vì bị đếm hai lần). Viết theo cặp `(số 2, số 5)` dùng tuple.

In [9]:
def four_directions(grid):
    m, n = len(grid), len(grid[0])
    cell = [[(factor_count(x, 2), factor_count(x, 5)) for x in row] for row in grid]
    Z = (0, 0)
    def add(a, b): return (a[0] + b[0], a[1] + b[1])
    L = [[Z] * n for _ in range(m)]; Rg = [[Z] * n for _ in range(m)]
    U = [[Z] * n for _ in range(m)]; D = [[Z] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            L[i][j] = add(L[i][j - 1] if j else Z, cell[i][j])
            U[i][j] = add(U[i - 1][j] if i else Z, cell[i][j])
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            Rg[i][j] = add(Rg[i][j + 1] if j + 1 < n else Z, cell[i][j])
            D[i][j] = add(D[i + 1][j] if i + 1 < m else Z, cell[i][j])
    best = 0
    for i in range(m):
        for j in range(n):
            c = cell[i][j]
            for h in (L[i][j], Rg[i][j]):
                for v in (U[i][j], D[i][j]):
                    best = max(best, min(h[0] + v[0] - c[0], h[1] + v[1] - c[1]))
    return best

for g, exp in examples:
    got = four_directions(g)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp cách 3:** O(m·n), bộ nhớ O(m·n) (4 mảng tuple). Dễ đọc hơn vì không có hiệu chỉ số, nhưng tốn bộ nhớ và chậm hơn do tạo nhiều tuple.

### 7.3 Cách 4 — Chuyển vị để chỉ cần xử lý "cánh ngang trái/phải"

Mẹo tổ chức code: chữ L luôn gồm một cánh ngang và một cánh dọc. Ta tính một lần trên lưới gốc; không cần chuyển vị (vì đã xét cả 4 tổ hợp). Nhưng nếu đề cho phép **đường thẳng đi qua ô góc theo cả hai phía** (biến thể "chữ T"), ta chỉ cần thay đổi tổ hợp. Cách 4 viết lại cách 2 gọn hơn bằng cách gộp `(2, 5)` vào một vòng lặp theo `p ∈ {2, 5}`:

In [10]:
def compact(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    tabs = []
    for p in (2, 5):
        c = [[factor_count(x, p) for x in row] for row in grid]
        R = [[0] * (n + 1) for _ in range(m)]; Cc = [[0] * n for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                R[i][j + 1] = R[i][j] + c[i][j]; Cc[i + 1][j] = Cc[i][j] + c[i][j]
        tabs.append((c, R, Cc))
    for i in range(m):
        for j in range(n):
            arms = []
            for c, R, Cc in tabs:
                arms.append((R[i][j], R[i][n] - R[i][j + 1], Cc[i][j], Cc[m][j] - Cc[i + 1][j], c[i][j]))
            (l2, r2, u2, d2, x2), (l5, r5, u5, d5, x5) = arms
            best = max(best, min(l2 + u2 + x2, l5 + u5 + x5), min(l2 + d2 + x2, l5 + d5 + x5),
                       min(r2 + u2 + x2, r5 + u5 + x5), min(r2 + d2 + x2, r5 + d5 + x5))
    return best

for g, exp in examples:
    print(compact(g), "OK" if compact(g) == exp else "SAI")

3 OK
0 OK


## 8. TRACE từng bước

### 8.1 Bảng số thừa số (2, 5) của ví dụ 1

In [11]:
g = examples[0][0]
for row in g:
    print("  ".join(f"{x:>3}:({factor_count(x, 2)},{factor_count(x, 5)})" for x in row))

 23:(0,0)   17:(0,0)   15:(0,1)    3:(0,0)   20:(2,1)
  8:(3,0)    1:(0,0)   20:(2,1)   27:(0,0)   11:(0,0)
  9:(0,0)    4:(2,0)    6:(1,0)    2:(1,0)   21:(0,0)
 40:(3,1)    9:(0,0)    1:(0,0)   10:(1,1)    6:(1,0)
 22:(1,0)    7:(0,0)    4:(2,0)    5:(0,1)    3:(0,0)


### 8.2 Trace cách 2: với mỗi ô in 4 giá trị chữ L và ô đạt max

In [12]:
def trace_prefix(grid):
    m, n = len(grid), len(grid[0])
    c2 = [[factor_count(x, 2) for x in row] for row in grid]; c5 = [[factor_count(x, 5) for x in row] for row in grid]
    R2 = [[0] * (n + 1) for _ in range(m)]; R5 = [[0] * (n + 1) for _ in range(m)]
    C2 = [[0] * n for _ in range(m + 1)]; C5 = [[0] * n for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            R2[i][j + 1] = R2[i][j] + c2[i][j]; R5[i][j + 1] = R5[i][j] + c5[i][j]
            C2[i + 1][j] = C2[i][j] + c2[i][j]; C5[i + 1][j] = C5[i][j] + c5[i][j]
    best, where = 0, None
    for i in range(m):
        line = []
        for j in range(n):
            h = [(R2[i][j], R5[i][j]), (R2[i][n] - R2[i][j + 1], R5[i][n] - R5[i][j + 1])]
            v = [(C2[i][j], C5[i][j]), (C2[m][j] - C2[i + 1][j], C5[m][j] - C5[i + 1][j])]
            vals = [min(a[0] + b[0] + c2[i][j], a[1] + b[1] + c5[i][j]) for a in h for b in v]
            line.append("".join(map(str, vals)))
            if max(vals) > best: best, where = max(vals), (i, j, vals.index(max(vals)))
        print(f"hàng {i}: " + " ".join(line) + "   (mỗi ô: TrTr TrDu PhTr PhDu)")
    names = ["trái+trên", "trái+dưới", "phải+trên", "phải+dưới"]
    print(f"max = {best} tại ô {where[:2]} dạng {names[where[2]]}")
    return best

trace_prefix(examples[0][0])

hàng 0: 0123 0022 0223 0213 2211   (mỗi ô: TrTr TrDu PhTr PhDu)
hàng 1: 0112 0011 2121 1302 2110   (mỗi ô: TrTr TrDu PhTr PhDu)
hàng 2: 0101 0000 2020 0202 1010   (mỗi ô: TrTr TrDu PhTr PhDu)
hàng 3: 1122 1111 3131 2312 3210   (mỗi ô: TrTr TrDu PhTr PhDu)
hàng 4: 1021 0011 2031 2120 2110   (mỗi ô: TrTr TrDu PhTr PhDu)
max = 3 tại ô (0, 0) dạng phải+dưới


3

Đọc bảng: mỗi ô có 4 chữ số ứng với 4 dạng chữ L. Đường minh họa của đề `15 → 20 → 6 → 1` (đi xuống cột 2) rồi quẹo phải sang `10` chính là chữ L có ô góc `(3, 2)` dạng phải+trên **bị cắt ngắn** (cánh phải mới tới `10`); bản kéo tới biên thêm ô `6` ở cuối hàng, giá trị vẫn ≥ 3 như quan sát 1 dự đoán. Có nhiều đường cùng đạt 3; trace in ô đầu tiên tìm thấy.

### 8.3 Dựng lại đường tối ưu cụ thể và kiểm tra bằng tích thật

In [13]:
def best_path(grid):
    m, n = len(grid), len(grid[0]); best = (-1, None)
    for i in range(m):
        for j in range(n):
            for hname, hcells in (("trái", [(i, c) for c in range(j)]), ("phải", [(i, c) for c in range(j + 1, n)])):
                for vname, vcells in (("trên", [(r, j) for r in range(i)]), ("dưới", [(r, j) for r in range(i + 1, m)])):
                    cells = hcells + [(i, j)] + vcells
                    a = sum(factor_count(grid[r][c], 2) for r, c in cells); b = sum(factor_count(grid[r][c], 5) for r, c in cells)
                    if min(a, b) > best[0]: best = (min(a, b), (i, j, hname, vname, cells))
    return best

z, (i, j, hn, vn, cells) = best_path(examples[0][0])
prod = 1
for r, c in cells: prod *= examples[0][0][r][c]
print(f"ô góc {(i, j)}, {hn}+{vn}, {len(cells)} ô, tích = {prod}, số 0 tận cùng = {trailing_zeros(prod)} (= {z})")

ô góc (0, 0), phải+dưới, 9 ô, tích = 22296384000, số 0 tận cùng = 3 (= 3)


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def maxTrailingZeros(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])

        def cnt(x, p):
            c = 0
            while x % p == 0:
                x //= p
                c += 1
            return c

        c2 = [[cnt(x, 2) for x in row] for row in grid]
        c5 = [[cnt(x, 5) for x in row] for row in grid]
        R2 = [[0] * (n + 1) for _ in range(m)]
        R5 = [[0] * (n + 1) for _ in range(m)]
        C2 = [[0] * n for _ in range(m + 1)]
        C5 = [[0] * n for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                R2[i][j + 1] = R2[i][j] + c2[i][j]
                R5[i][j + 1] = R5[i][j] + c5[i][j]
                C2[i + 1][j] = C2[i][j] + c2[i][j]
                C5[i + 1][j] = C5[i][j] + c5[i][j]
        best = 0
        for i in range(m):
            for j in range(n):
                x2, x5 = c2[i][j], c5[i][j]                                  # ô góc (đếm 1 lần)
                l2, l5 = R2[i][j], R5[i][j]                                  # cánh trái
                r2, r5 = R2[i][n] - R2[i][j + 1], R5[i][n] - R5[i][j + 1]    # cánh phải
                u2, u5 = C2[i][j], C5[i][j]                                  # cánh trên
                d2, d5 = C2[m][j] - C2[i + 1][j], C5[m][j] - C5[i + 1][j]    # cánh dưới
                best = max(best,
                           min(l2 + u2 + x2, l5 + u5 + x5),
                           min(l2 + d2 + x2, l5 + d5 + x5),
                           min(r2 + u2 + x2, r5 + u5 + x5),
                           min(r2 + d2 + x2, r5 + d5 + x5))
        return best

In [15]:
sol = Solution()
for g, exp in examples:
    got = sol.maxTrailingZeros(g)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

Gọi một đường có góc là `P`, `a(P)` và `b(P)` là tổng số thừa số 2 và 5 trên `P`; số 0 tận cùng của tích là `min(a(P), b(P))` (mục 3.1).

**Bổ đề 1 (cấu trúc).** Mọi `P` là: ô góc `(i, j)` + một đoạn ngang liền kề `(i, j)` về **một phía** + một đoạn dọc liền kề `(i, j)` về **một phía** (mỗi đoạn có thể rỗng). Nếu `P` có một lần quẹo, ô góc là ô quẹo; nếu `P` thẳng, chọn ô góc là một đầu mút và đoạn còn lại rỗng.

**Bổ đề 2 (kéo dài không thiệt).** Thay đoạn ngang của `P` bằng **toàn bộ** phía đó tới biên được đường `P'` ⊇ `P` (vẫn là đường có góc hợp lệ, không lặp ô vì cánh ngang và cánh dọc chỉ giao nhau ở ô góc). Vì mọi ô có số thừa số không âm: `a(P') ≥ a(P)`, `b(P') ≥ b(P)` ⇒ `min` không giảm. Tương tự cho đoạn dọc.

⇒ Tồn tại đường tối ưu thuộc tập 4·m·n chữ L "đầy đủ". Thuật toán duyệt đúng tập này và tính `a, b` của mỗi chữ L chính xác bằng tiền tố (ô góc cộng một lần) ⇒ trả về đúng max. ∎

Lưu ý chữ L "đầy đủ" có thể có một cánh rỗng khi ô góc nằm trên biên — trường hợp đó tự nhiên trả về tổng 0 từ tiền tố.

## 11. Phân tích Time / Space Complexity

`N = m·n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi đường | O(N·(m+n)²) | O(N) |
| 1b. 4 chữ L cộng trực tiếp | O(N·(m+n)) | O(N) |
| 2. Tiền tố hàng/cột | O(N) | O(N) |
| 3. Bốn mảng hướng | O(N) | O(N) (4 mảng tuple) |
| 4. Gọn (gộp p) | O(N) | O(N) |

In [16]:
rng = random.Random(2245)
for m, n in [(316, 316), (1, 100000), (100000, 1)]:
    big = [[rng.randint(1, 1000) for _ in range(n)] for _ in range(m)]
    t = time.perf_counter(); r = Solution().maxTrailingZeros(big); dt = time.perf_counter() - t
    print(f"{m}x{n}: {r} số 0, {dt*1000:.0f} ms")

316x316: 200 số 0, 73 ms


1x100000: 24773 số 0, 77 ms


100000x1: 24888 số 0, 314 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "four_L_direct": four_L_direct, "prefix_solution": prefix_solution,
       "four_directions": four_directions, "compact": compact, "Solution": Solution().maxTrailingZeros}
edge = examples + [
    ([[10]], 1), ([[7]], 0), ([[1000]], 3),
    ([[2, 5]], 1), ([[2], [5]], 1),
    ([[2, 1], [1, 5]], 1),                 # (0,0) phải… 2·1 rồi xuống 5: chữ L
    ([[2, 1, 5]], 1),                      # đường thẳng xuyên hàng
    ([[4, 1], [1, 25]], 2),                # 4 · 1 · 25 = 100 qua góc
    ([[5, 5, 5], [2, 1, 2], [2, 2, 2]], 2),
    ([[1] * 4 for _ in range(4)], 0),
]
for g, exp in edge:
    for name, f in ALL.items():
        gg = [row[:] for row in g]
        assert f(gg) == exp, (name, g, f(g), exp)
        assert gg == g
print("Edge case OK:", len(edge))

Edge case OK: 12


Các kỳ vọng tự dựng ở trên được tính tay và đồng thời xác nhận bằng `brute` (liệt kê thật mọi đường). Ví dụ `[[5,5,5],[2,1,2],[2,2,2]]`: hàng đầu có ba số 5, cột đầu có `5, 2, 2`. Chữ L ô góc `(0, 0)` phải+dưới = `5·5·5·2·2 = 500` ⇒ (2: 2, 5: 3) ⇒ **2**. Có thể nghĩ "lấy ba số 5 và ba số 2" để được 3 — nhưng ba số 2 nằm ở hàng cuối, không có chữ L nào chứa cả hàng đầu lẫn hàng cuối (cần hai lần quẹo). Cell dưới in đường tối ưu mà máy tìm được:

In [18]:
z, info = best_path([[5, 5, 5], [2, 1, 2], [2, 2, 2]])
print("max =", z, "| dạng:", info[2] + "+" + info[3], "ô góc", info[:2], "các ô:", info[4])

max = 2 | dạng: phải+dưới ô góc (0, 0) các ô: [(0, 1), (0, 2), (0, 0), (1, 0), (2, 0)]


Bài học: hình dạng chữ L giới hạn rất chặt tập ô có thể gom — tính tay rất dễ "gom quá tay", nên luôn đối chiếu với bộ liệt kê.

### 12.2 Đối chiếu ngẫu nhiên với liệt kê thật (lưới nhỏ, seed cố định)

Chọn giá trị từ tập có nhiều thừa số 2 và 5 để đáp án đa dạng.

In [19]:
rng = random.Random(42)
pool = [1, 2, 4, 5, 8, 10, 16, 20, 25, 3, 7, 50, 125, 1000, 625, 512]
for _ in range(500):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.choice(pool) for _ in range(n)] for _ in range(m)]
    ref = brute(g)
    for name, f in ALL.items():
        assert f(g) == ref, (name, g)
print("500 case ngẫu nhiên OK")

500 case ngẫu nhiên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng số 0 tận cùng của từng số

In [20]:
def wrong_sum_zeros(grid):
    m, n = len(grid), len(grid[0]); best = 0
    for i in range(m):
        for j in range(n):
            for h in (grid[i][:j], grid[i][j + 1:]):
                for v in ([grid[r][j] for r in range(i)], [grid[r][j] for r in range(i + 1, m)]):
                    best = max(best, sum(trailing_zeros(x) for x in h + v + [grid[i][j]]))   # SAI
    return best

g = [[2, 5]]
print("bản sai :", wrong_sum_zeros(g), "| bản đúng:", Solution().maxTrailingZeros(g))

bản sai : 0 | bản đúng: 1


**Vì sao sai:** `2` và `5` đều không có số 0 tận cùng, nhưng tích `10` có. Số 0 sinh ra từ **ghép cặp 2 và 5 trên toàn đường**. **Sửa:** đếm tổng số 2, tổng số 5, rồi lấy `min`.

### Lỗi 2 — Lấy `min` cho từng cánh rồi cộng

In [21]:
def wrong_min_per_arm(grid):
    m, n = len(grid), len(grid[0]); best = 0
    def c(vals): return sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals)
    for i in range(m):
        for j in range(n):
            x = c([grid[i][j]])
            for h in (c(grid[i][:j]), c(grid[i][j + 1:])):
                for v in (c([grid[r][j] for r in range(i)]), c([grid[r][j] for r in range(i + 1, m)])):
                    best = max(best, min(h) + min(v) + min(x))                          # SAI
    return best

g = [[4, 1], [1, 25]]
print("bản sai :", wrong_min_per_arm(g), "| bản đúng:", Solution().maxTrailingZeros(g))

bản sai : 0 | bản đúng: 2


**Vì sao sai:** cánh `4` có (2, 0), cánh `25` có (0, 2) — mỗi cánh riêng có `min = 0`, nhưng cả đường có (2, 2) ⇒ 2. `min` không cộng tính được: `min(a + c, b + d) ≥ min(a, b) + min(c, d)`. **Sửa:** cộng riêng số 2 và số 5 trên cả đường rồi mới `min`.

### Lỗi 3 — Đếm ô góc hai lần

In [22]:
def wrong_double_corner(grid):
    m, n = len(grid), len(grid[0]); best = 0
    def c(vals): return sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals)
    for i in range(m):
        for j in range(n):
            for h in (c(grid[i][:j + 1]), c(grid[i][j:])):                               # gồm ô góc
                for v in (c([grid[r][j] for r in range(i + 1)]), c([grid[r][j] for r in range(i, m)])):  # lại gồm ô góc
                    best = max(best, min(h[0] + v[0], h[1] + v[1]))                     # SAI: không trừ ô góc
    return best

g = [[10]]
print("bản sai :", wrong_double_corner(g), "| bản đúng:", Solution().maxTrailingZeros(g))

bản sai : 2 | bản đúng: 1


**Sửa:** nếu cả hai cánh "bao gồm" ô góc thì trừ đi một lần (như cách 3), hoặc định nghĩa cánh **không** gồm ô góc rồi cộng ô góc riêng (như cách 2).

### Lỗi 4 — Chỉ xét đường thẳng (quên chỗ quẹo)

In [23]:
def wrong_straight_only(grid):
    m, n = len(grid), len(grid[0]); best = 0
    def z(vals):
        return min(sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals))
    for row in grid: best = max(best, z(row))
    for j in range(n): best = max(best, z([grid[i][j] for i in range(m)]))
    return best

g = [[2, 1], [1, 5]]
print("bản sai :", wrong_straight_only(g), "| bản đúng:", Solution().maxTrailingZeros(g))

bản sai : 0 | bản đúng: 1


**Sửa:** xét cả 4 dạng chữ L tại mọi ô.

### Lỗi 5 — Cho phép "chữ T" hoặc "dấu cộng" (hai cánh cùng phương)

In [24]:
def wrong_plus_shape(grid):
    m, n = len(grid), len(grid[0]); best = 0
    def c(vals): return sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals)
    for i in range(m):
        for j in range(n):
            a = c(grid[i]); b = c([grid[r][j] for r in range(m) if r != i])          # SAI: cả hàng + cả cột
            best = max(best, min(a[0] + b[0], a[1] + b[1]))
    return best

g = [[2, 1, 2], [1, 5, 1], [1, 5, 1]]
print("bản sai :", wrong_plus_shape(g), "| bản đúng:", Solution().maxTrailingZeros(g))

bản sai : 2 | bản đúng: 1


**Vì sao sai:** dấu cộng có ba cánh trở lên (phải đi qua ô góc hai lần hoặc quẹo hai lần) — không phải đường có góc. Ở đây bản sai ghép `2·1·2` (cả hàng) với `5·5` (cột dưới) ⇒ 2 số 0, trong khi đường hợp lệ tốt nhất chỉ dùng một phía của hàng. **Sửa:** cánh ngang chỉ **một phía**, cánh dọc chỉ **một phía**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số chữ số 0 tận cùng của tích" ⇒ đếm **thừa số 2 và 5**, cộng dồn, `min` ở cuối.
- Giá trị dương + "tối đa hóa một hàm không giảm theo tập ô" ⇒ đường/đoạn tối ưu **kéo tới biên**.
- Đoạn liên tiếp trên hàng/cột ⇒ **tiền tố theo hàng và theo cột**.
- Nhiều dạng hình học (L, T, chữ thập) ⇒ liệt kê rõ các **cánh** và tổ hợp hợp lệ; đếm ô góc đúng một lần.
- Bài cùng họ: 172 Factorial Trailing Zeroes, 2117 Abbreviating the Product of a Range, 1878, 304 Range Sum Query 2D.

## 15. Biến thể của bài

### 15.1 Biến thể A — Chỉ cho phép đường **thẳng** (không quẹo)

Chỉ cần tổng cả hàng và cả cột (kéo tới biên hai phía).

In [25]:
def straight_only(grid):
    return wrong_straight_only(grid)          # đúng cho biến thể này

def straight_brute(grid):
    m, n = len(grid), len(grid[0]); best = 0
    for i in range(m):
        for l in range(n):
            for r in range(l, n):
                vals = grid[i][l:r + 1]
                best = max(best, min(sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals)))
    for j in range(n):
        col = [grid[i][j] for i in range(m)]
        for t in range(m):
            for b in range(t, m):
                vals = col[t:b + 1]
                best = max(best, min(sum(factor_count(x, 2) for x in vals), sum(factor_count(x, 5) for x in vals)))
    return best

rng = random.Random(151)
for _ in range(300):
    g = [[rng.choice(pool) for _ in range(rng.randint(1, 4))]]
    g += [[rng.choice(pool) for _ in range(len(g[0]))] for _ in range(rng.randint(0, 3))]
    assert straight_only(g) == straight_brute(g)
print("đường thẳng: kéo tới biên hai phía là đủ — khớp brute force")

đường thẳng: kéo tới biên hai phía là đủ — khớp brute force


### 15.2 Biến thể B — Giá trị có thể bằng **0** (ô số 0)

Ô 0 làm tích bằng 0 — theo quy ước "0 có vô số chữ số 0" sẽ phá vỡ bài; còn nếu cấm đi qua ô 0 thì quan sát "kéo tới biên" sai (phải dừng trước ô 0). Khi đó mỗi cánh kéo tới **ô 0 gần nhất** (hoặc biên): tính bằng tiền tố + vị trí 0 gần nhất mỗi hướng. Minh họa cánh trái tối đa:

In [26]:
def left_arm_limits(row):
    # với mỗi j: cánh trái hợp lệ dài nhất kết thúc ngay trước j, dừng trước ô 0 gần nhất
    out = []; last_zero = -1
    for j, x in enumerate(row):
        out.append((last_zero + 1, j - 1))   # đoạn [l, j-1] (rỗng nếu l > j-1)
        if x == 0: last_zero = j
    return out

print(left_arm_limits([3, 0, 4, 5, 0, 2]))

[(0, -1), (0, 0), (2, 1), (2, 2), (2, 3), (5, 4)]


### 15.3 Biến thể C — Số chữ số 0 tận cùng trong **hệ cơ số 6** (`6 = 2·3`)

Chỉ cần thay thừa số 5 bằng thừa số 3 — khung thuật toán giữ nguyên.

In [27]:
def max_trailing_base6(grid):
    m, n = len(grid), len(grid[0]); best = 0
    a2 = [[factor_count(x, 2) for x in row] for row in grid]; a3 = [[factor_count(x, 3) for x in row] for row in grid]
    for i in range(m):
        for j in range(n):
            for hl in (range(j), range(j + 1, n)):
                for vl in (range(i), range(i + 1, m)):
                    s2 = a2[i][j] + sum(a2[i][c] for c in hl) + sum(a2[r][j] for r in vl)
                    s3 = a3[i][j] + sum(a3[i][c] for c in hl) + sum(a3[r][j] for r in vl)
                    best = max(best, min(s2, s3))
    return best

print(max_trailing_base6([[4, 3, 2], [7, 6, 1], [8, 8, 8]]), "(ví dụ 2 trong cơ số 6 — có thừa số 3 nên khác 0)")

2 (ví dụ 2 trong cơ số 6 — có thừa số 3 nên khác 0)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê mọi đường | đúng định nghĩa | O(N(m+n)²) | O(N) | kiểm chứng |
| 1b. 4 chữ L, cộng trực tiếp | kéo tới biên | O(N(m+n)) | O(N) | bước trung gian |
| 2. Tiền tố hàng/cột | hiệu tiền tố O(1) | O(N) | O(N) | **mặc định** |
| 3. Bốn mảng hướng | tích lũy gồm ô, trừ ô góc | O(N) | O(N) | dễ đọc |
| 4. Gọn | gộp 2 và 5 | O(N) | O(N) | code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxTrailingZeros([[10, 10], [10, 10]])` và `maxTrailingZeros([[2, 2, 2], [5, 1, 1]])`?

**Bài 2 (tính tay):** Với lưới `[[8, 1], [1, 125]]`, liệt kê 4 chữ L tại ô `(0, 1)` và số 0 tận cùng của mỗi chữ.

**Bài 3 (điền TODO):** Viết `bt3_todo(grid)` trả về ma trận `best[i][j]` = số 0 tận cùng lớn nhất trong 4 chữ L tại ô `(i, j)`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(grid)` cài cách 3 (bốn mảng hướng) không nhìn code.

**Bài 5 (mini challenge):** Cho phép **chữ T** (một cánh dọc một phía + cả hàng hai phía qua ô góc — không phải đường có góc, chỉ là một tập ô). Viết `bt5_todo(grid)` trả về số 0 tận cùng lớn nhất của tích các ô trong một hình chữ T bất kỳ (các cánh có thể rỗng, đều kéo tới biên).

In [28]:
def bt3_todo(grid):
    # TODO: ma trận best[i][j]
    pass

def bt4_todo(grid):
    # TODO: bốn mảng hướng
    pass

def bt5_todo(grid):
    # TODO: hình chữ T
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: mỗi ô (1, 1). Mức 2: chữ L dài nhất có 3 ô ⇒ (3, 3). Mức 3: `3`; lưới hai: ô góc `(0, 0)` phải+dưới = `2·2·2·5` ⇒ (3, 1) ⇒ 1; đáp án `1`.

**Bài 2.** Mức 1: ô `(0, 1)` giá trị 1; cánh trái `8`, cánh phải rỗng; cánh trên rỗng, cánh dưới `125`. Mức 2: trái+dưới `8·1·125 = 1000` ⇒ 3. Mức 3: trái+trên `8` ⇒ 0; phải+dưới `125` ⇒ 0; phải+trên `1` ⇒ 0.

**Bài 3.** Mức 1: tái dùng tiền tố cách 2. Mức 2: lưu `max` của 4 giá trị vào `best[i][j]`. Mức 3: `max` của ma trận chính là đáp án bài gốc.

**Bài 4.** Mức 1: `L[i][j] = L[i][j−1] + ô`. Mức 2: `Rg, U, D` tương tự theo hướng ngược. Mức 3: chữ L = `h + v − ô`.

**Bài 5.** Mức 1: chữ T = cả hàng `i` + cánh trên hoặc dưới của cột `j`. Mức 2: dùng tiền tố hàng (cả hàng = `R[i][n]`) và tiền tố cột. Mức 3: cũng xét chữ T "nằm ngang" (cả cột + một phía hàng) — 4 hình mỗi ô.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `3`, `1`. **Bài 2:** trái+dưới = 3, còn lại 0.

In [29]:
assert Solution().maxTrailingZeros([[10, 10], [10, 10]]) == 3
assert Solution().maxTrailingZeros([[2, 2, 2], [5, 1, 1]]) == 1

def bt3(grid):
    m, n = len(grid), len(grid[0])
    c2 = [[factor_count(x, 2) for x in row] for row in grid]; c5 = [[factor_count(x, 5) for x in row] for row in grid]
    R2 = [[0] * (n + 1) for _ in range(m)]; R5 = [[0] * (n + 1) for _ in range(m)]
    C2 = [[0] * n for _ in range(m + 1)]; C5 = [[0] * n for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            R2[i][j + 1] = R2[i][j] + c2[i][j]; R5[i][j + 1] = R5[i][j] + c5[i][j]
            C2[i + 1][j] = C2[i][j] + c2[i][j]; C5[i + 1][j] = C5[i][j] + c5[i][j]
    best = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            hs = [(R2[i][j], R5[i][j]), (R2[i][n] - R2[i][j + 1], R5[i][n] - R5[i][j + 1])]
            vs = [(C2[i][j], C5[i][j]), (C2[m][j] - C2[i + 1][j], C5[m][j] - C5[i + 1][j])]
            best[i][j] = max(min(h[0] + v[0] + c2[i][j], h[1] + v[1] + c5[i][j]) for h in hs for v in vs)
    return best

def bt4(grid):
    return four_directions(grid)

def bt5(grid):
    m, n = len(grid), len(grid[0]); best = 0
    c = [[(factor_count(x, 2), factor_count(x, 5)) for x in row] for row in grid]
    def tot(cells): return (sum(c[r][q][0] for r, q in cells), sum(c[r][q][1] for r, q in cells))
    for i in range(m):
        for j in range(n):
            row = [(i, q) for q in range(n)]; col = [(r, j) for r in range(m)]
            shapes = [row + [(r, j) for r in range(i)], row + [(r, j) for r in range(i + 1, m)],
                      col + [(i, q) for q in range(j)], col + [(i, q) for q in range(j + 1, n)]]
            for s in shapes:
                a, b = tot(s); best = max(best, min(a, b))
    return best

assert bt3([[8, 1], [1, 125]])[0][1] == 3
rng = random.Random(1945)
for _ in range(300):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.choice(pool) for _ in range(n)] for _ in range(m)]
    ref = brute(g)
    assert max(max(r) for r in bt3(g)) == ref and bt4(g) == ref
    assert bt5(g) >= ref                       # chữ T chứa chữ L đầy đủ nên không kém hơn
print("Đáp án bài tập: tất cả assert OK")
print("ví dụ chữ T:", bt5([[2, 1, 2], [1, 5, 1], [1, 5, 1]]), "vs đường có góc:", Solution().maxTrailingZeros([[2, 1, 2], [1, 5, 1], [1, 5, 1]]))

Đáp án bài tập: tất cả assert OK
ví dụ chữ T: 2 vs đường có góc: 1


Ghi chú bài 5: mọi chữ L đầy đủ đều là **tập con** của một chữ T (thêm phía còn lại của hàng hoặc cột), nên theo tính đơn điệu, đáp án chữ T ≥ đáp án bài gốc — assert ở trên kiểm tra tính chất này.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# c2, c5: số thừa số 2, 5 mỗi ô; R*: tiền tố hàng (n+1 cột), C*: tiền tố cột (m+1 hàng)
for i, j:
    x = (c2[i][j], c5[i][j])                       # ô góc, cộng 1 lần
    L = (R2[i][j], R5[i][j]);  Rt = (R2[i][n]-R2[i][j+1], R5[i][n]-R5[i][j+1])
    U = (C2[i][j], C5[i][j]);  D  = (C2[m][j]-C2[i+1][j], C5[m][j]-C5[i+1][j])
    best = max(best, *(min(h[0]+v[0]+x[0], h[1]+v[1]+x[1]) for h in (L, Rt) for v in (U, D)))
```

**Checklist:**
- [ ] Đếm **thừa số 2 và 5**, không nhân thật, không cộng số 0 của từng số.
- [ ] `min` lấy **sau khi** cộng toàn đường.
- [ ] Ô góc tính đúng **một** lần.
- [ ] Mỗi cánh chỉ **một phía**, kéo **tới biên**.
- [ ] Đã thử: lưới 1×1, một hàng, một cột, chữ L tách 2 và 5 ở hai cánh.